Personalized ChronoRoPE — Kaggle Experiment
==============================================
User-Adaptive Temporal Warping for Sequential Recommendation

This script is self-contained and runnable on Kaggle with GPU.
Compares:
  1. SASRec (Baseline) — Absolute Positional Encoding, no time
  2. SASRec + RoTE — Multi-level Rotary Time Embedding (SIGIR'26)
  3. SASRec + ChronoRoPE — Personalized temporal warping (Ours)

Usage on Kaggle:
  - Create a new Kaggle Notebook
  - Enable GPU (Settings → Accelerator → GPU T4 x2)
  - Upload this file or paste the code
  - Run All

Usage locally:
  python chrono_kaggle_experiment.py

In [ ]:
from __future__ import annotations

import gzip
import json
import math
import os
import random
import time
import urllib.request
from collections import defaultdict
from typing import NamedTuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## CONFIG

In [ ]:
CONFIG = {
    "seed": 2024,
    "maxlen": 50,
    "hidden_units": 64,
    "num_blocks": 2,
    "num_heads": 1,
    "dropout_rate": 0.5,
    "lr": 0.001,
    "batch_size": 512,
    "num_epochs": 500,       # 500 epochs for full convergence on Kaggle GPU
    "eval_interval": 10,
    "topk": [5, 10],
    "grad_clip": 1.0,
    "patience": 60,
    "full_eval": True,       # Full-catalog evaluation (ranks target against all ~12k items, matching SIGIR'26 paper)
    "num_neg_eval": 99,
    # RoTE
    "year_base": 1_000_000.0,
    "month_base": 10_000.0,
    "day_base": 100.0,
    "year_weight": 1.5,
    "month_weight": 1.0,
    "day_weight": 0.5,
    # ChronoRoPE
    "meta_input_dim": 9,
    "meta_hidden_dims": [32, 16],
    "meta_eps": 0.1,
    "meta_init_alpha": 1.0,
    "rope_base": 10_000.0,
    "diversity_lambda": 0.01,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## SECTION 1: DATA LOADING

In [ ]:
class SequentialDataset(NamedTuple):
    user_seqs: dict[int, list[int]]
    user_times: dict[int, list[float]]
    user_features: dict[int, np.ndarray]
    num_users: int
    num_items: int


def download_amazon_dataset(category: str = "Toys_and_Games", data_dir: str = "data"):
    """Download Amazon Reviews 2014 (5-core) dataset."""
    os.makedirs(data_dir, exist_ok=True)
    output_path = os.path.join(data_dir, f"{category}_5_time.txt")

    if os.path.exists(output_path):
        print(f"  Dataset already exists: {output_path}")
        return output_path

    url = f"https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_{category}_5.json.gz"
    gz_path = os.path.join(data_dir, f"reviews_{category}_5.json.gz")

    print(f"  Downloading {url} ...")
    urllib.request.urlretrieve(url, gz_path)
    print(f"  Downloaded to {gz_path}")

    # Parse and write tab-separated file
    print(f"  Parsing reviews ...")
    with gzip.open(gz_path, "rt", encoding="utf-8") as f, open(output_path, "w") as out:
        for line in f:
            try:
                review = json.loads(line.strip().replace("'", '"'))
            except json.JSONDecodeError:
                # Fallback: use eval for Python-formatted lines
                review = eval(line.strip())
            user = review["reviewerID"]
            item = review["asin"]
            ts = int(review["unixReviewTime"])
            out.write(f"{user}\t{item}\t{ts}\n")

    print(f"  Saved to {output_path}")
    return output_path


def compute_user_temporal_features(
    user_times: dict[int, list[float]], recent_k: int = 10,
) -> dict[int, np.ndarray]:
    """Compute 7 temporal features per user for Meta-MLP."""
    user_features: dict[int, np.ndarray] = {}
    all_last = [t[-1] for t in user_times.values() if t]
    max_time = max(all_last) if all_last else 1.0

    for uid, times in user_times.items():
        n = len(times)
        if n < 2:
            feat = np.zeros(9, dtype=np.float32)
            feat[5] = np.log1p(n)
            user_features[uid] = feat
            continue

        span = max(times[-1] - times[0], 1.0)
        gaps = np.diff(times).astype(np.float64) / 3600.0  # hours

        k = min(recent_k, n)
        r_times = times[-k:]
        r_span = max(r_times[-1] - r_times[0], 1.0)

        user_features[uid] = np.array([
            np.log1p(n / (span / 3600.0)),         # global density
            np.log1p(k / (r_span / 3600.0)),        # recent density
            np.log1p(np.mean(gaps)),                 # mean gap
            np.log1p(np.std(gaps)),                  # std gap
            np.log1p(np.max(gaps)),                  # max gap
            np.log1p(n),                             # seq length
            times[-1] / max_time,                    # recency
            np.sin(2 * np.pi * ((times[-1] % 86400) / 3600.0) / 24.0), # circ_sin
            np.cos(2 * np.pi * ((times[-1] % 86400) / 3600.0) / 24.0), # circ_cos
        ], dtype=np.float32)

    # Standardize features (Z-score normalization)
    feats = np.stack(list(user_features.values()))
    mean = np.mean(feats, axis=0, keepdims=True)
    std = np.std(feats, axis=0, keepdims=True) + 1e-8
    
    for uid in user_features:
        user_features[uid] = (user_features[uid] - mean[0]) / std[0]

    return user_features


def load_data(filepath: str) -> SequentialDataset:
    """Load tab-separated data file."""
    raw: dict[str, list[tuple[str, float]]] = defaultdict(list)
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            raw[parts[0]].append((parts[1], float(parts[2])))

    for u in raw:
        raw[u].sort(key=lambda x: x[1])

    user2idx, item2idx = {}, {}
    user_seqs, user_times = {}, {}

    for raw_u, interactions in raw.items():
        if raw_u not in user2idx:
            user2idx[raw_u] = len(user2idx) + 1
        uid = user2idx[raw_u]
        items, times = [], []
        first_ts = interactions[0][1] if interactions else 0.0
        for raw_i, ts in interactions:
            if raw_i not in item2idx:
                item2idx[raw_i] = len(item2idx) + 1
            items.append(item2idx[raw_i])
            times.append(ts - first_ts)
        user_seqs[uid] = items
        user_times[uid] = times

    # BUG-3 FIX: Compute features from train-only timestamps (exclude val+test)
    train_times = {uid: t[:-2] if len(t) >= 3 else t for uid, t in user_times.items()}
    return SequentialDataset(
        user_seqs=user_seqs,
        user_times=user_times,
        user_features=compute_user_temporal_features(train_times),
        num_users=len(user2idx),
        num_items=len(item2idx),
    )

## SECTION 2: DATA SAMPLER

In [ ]:
class TrainSampler:
    def __init__(self, dataset: SequentialDataset, maxlen=50, batch_size=512, num_neg_eval=99):
        self.ds = dataset
        self.maxlen = maxlen
        self.batch_size = batch_size
        self.num_neg_eval = num_neg_eval
        self.user_sets = {u: set(items) for u, items in dataset.user_seqs.items()}
        self.users = [u for u, items in dataset.user_seqs.items() if len(items) >= 3]

    def _pad(self, seq, maxlen, val=0):
        return ([val] * max(0, maxlen - len(seq)) + seq)[-maxlen:]

    def _neg(self, uid):
        s = self.user_sets[uid]
        while True:
            n = random.randint(1, self.ds.num_items)
            if n not in s:
                return n

    def train_epoch(self):
        random.shuffle(self.users)
        for i in range(0, len(self.users), self.batch_size):
            batch = self.users[i : i + self.batch_size]
            logs, tms, poss, negs, feats = [], [], [], [], []
            for uid in batch:
                items, times = self.ds.user_seqs[uid], self.ds.user_times[uid]
                # BUG-1 FIX: Use train set only (items[:-2]), match official:
                #   seq = train[:-1], pos = train[1:]
                train_items = items[:-2]
                train_times = times[:-2]
                logs.append(self._pad(train_items[:-1], self.maxlen))
                tms.append(self._pad(train_times[:-1], self.maxlen, 0.0))
                pos = self._pad(train_items[1:], self.maxlen)
                poss.append(pos)
                negs.append([0 if p == 0 else self._neg(uid) for p in pos])
                feats.append(self.ds.user_features[uid])
            yield (
                torch.tensor(logs, dtype=torch.long),
                torch.tensor(tms, dtype=torch.float32),
                torch.tensor(poss, dtype=torch.long),
                torch.tensor(negs, dtype=torch.long),
                torch.tensor(np.stack(feats), dtype=torch.float32),
            )

    def eval_batches(self, mode="test", full_eval=True):
        for i in range(0, len(self.users), self.batch_size):
            batch = self.users[i : i + self.batch_size]
            logs, tms, feats, targets_or_cands = [], [], [], []
            uids = []
            for uid in batch:
                items, times = self.ds.user_seqs[uid], self.ds.user_times[uid]
                if mode == "val":
                    inp_i, inp_t, target = items[:-2], times[:-2], items[-2]
                    hist_set = set(items[:-2])
                else:
                    inp_i, inp_t, target = items[:-1], times[:-1], items[-1]
                    hist_set = set(items[:-1])
                logs.append(self._pad(inp_i, self.maxlen))
                tms.append(self._pad(inp_t, self.maxlen, 0.0))
                feats.append(self.ds.user_features[uid])
                uids.append(uid)
                if full_eval:
                    targets_or_cands.append((target, hist_set))
                else:
                    c = [target]
                    s = self.user_sets[uid]
                    max_neg = min(self.num_neg_eval, self.ds.num_items - len(s))
                    while len(c) < 1 + max_neg:
                        n = random.randint(1, self.ds.num_items)
                        if n not in s and n not in c:
                            c.append(n)
                    targets_or_cands.append(c)
            if full_eval:
                yield (
                    uids,
                    torch.tensor(logs, dtype=torch.long),
                    torch.tensor(tms, dtype=torch.float32),
                    torch.tensor(np.stack(feats), dtype=torch.float32),
                    targets_or_cands,
                )
            else:
                max_cand_len = max(len(c) for c in targets_or_cands)
                cands_padded = [c + [c[0]] * (max_cand_len - len(c)) for c in targets_or_cands]
                yield (
                    uids,
                    torch.tensor(logs, dtype=torch.long),
                    torch.tensor(tms, dtype=torch.float32),
                    torch.tensor(np.stack(feats), dtype=torch.float32),
                    torch.tensor(cands_padded, dtype=torch.long),
                )

## SECTION 3: MODULES

In [ ]:
# ── RoPE helpers ─────────────────────────────────────────────

SECONDS_PER_DAY = 86_400.0
DAYS_PER_YEAR = 365.25
DAYS_PER_MONTH = 30.4375


def decompose_unix_timestamp(ts):
    days = ts.float() / SECONDS_PER_DAY
    return days / DAYS_PER_YEAR, days / DAYS_PER_MONTH, days


def _apply_rotary(x, cos, sin):
    x_even, x_odd = x[..., ::2], x[..., 1::2]
    return torch.stack((x_even * cos - x_odd * sin, x_even * sin + x_odd * cos), dim=-1).flatten(-2)


# ── Meta-MLP ─────────────────────────────────────────────────

class MetaMLP(nn.Module):
    """Per-level α: outputs (α_y, α_m, α_d) for independent Y/M/D base scaling."""
    def __init__(self, input_dim=9, hidden_dims=None, eps=0.1, init_alpha=1.0):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [32, 16]
        self.eps = eps
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.GELU()]
            prev = h
        layers.append(nn.Linear(prev, 3))  # 3 outputs: α_y, α_m, α_d
        self.mlp = nn.Sequential(*layers)
        # Init near identity
        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        target = init_alpha - eps
        if target > 0:
            self.mlp[-1].bias.data.fill_(math.log(math.exp(target) - 1.0))

    def forward(self, x):
        return F.softplus(self.mlp(x)) + self.eps  # (batch, 3)

    @staticmethod
    def diversity_loss(alpha):
        return torch.tensor(0.0, device=alpha.device)


# ── Rotary Embeddings ────────────────────────────────────────

class RotaryTimeEmbedding(nn.Module):
    def __init__(self, dim, base=10_000.0):
        super().__init__()
        self.register_buffer("inv_freq", 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim)), persistent=False)

    def _cs(self, pos):
        f = torch.einsum("bs,d->bsd", pos.float(), self.inv_freq)
        return f.cos().unsqueeze(1), f.sin().unsqueeze(1)

    def forward(self, q, k, qp, kp):
        cq, sq = self._cs(qp)
        ck, sk = self._cs(kp)
        return _apply_rotary(q, cq, sq), _apply_rotary(k, ck, sk)


class MultiLevelRoTE(nn.Module):
    def __init__(self, dim, yb=1e6, mb=1e4, db=100.0, yw=1.5, mw=1.0, dw=0.5):
        super().__init__()
        idx = torch.arange(0, dim, 2).float() / dim
        self.register_buffer("iy", 1.0 / (yb ** idx), persistent=False)
        self.register_buffer("im", 1.0 / (mb ** idx), persistent=False)
        self.register_buffer("id", 1.0 / (db ** idx), persistent=False)
        self.yw, self.mw, self.dw = yw, mw, dw

    def precompute(self, ymd):
        y, m, d = ymd
        ty = torch.einsum("bs,d->bsd", y.float(), self.iy)
        tm = torch.einsum("bs,d->bsd", m.float(), self.im)
        td = torch.einsum("bs,d->bsd", d.float(), self.id)
        return tuple((t.cos().unsqueeze(1), t.sin().unsqueeze(1)) for t in (ty, tm, td))

    def apply(self, x, cache):
        (cy, sy), (cm, sm), (cd, sd) = cache
        return self.yw * _apply_rotary(x, cy, sy) + self.mw * _apply_rotary(x, cm, sm) + self.dw * _apply_rotary(x, cd, sd)

    def forward(self, q, k, qy, ky):
        qc = self.precompute(qy)
        kc = qc if qy is ky else self.precompute(ky)
        return self.apply(q, qc), self.apply(k, kc)

## SECTION 4: MODELS

In [ ]:
class PointWiseFeedForward(nn.Module):
    def __init__(self, d, dr):
        super().__init__()
        self.c1 = nn.Conv1d(d, d, 1)
        self.c2 = nn.Conv1d(d, d, 1)
        self.d1 = nn.Dropout(dr)
        self.d2 = nn.Dropout(dr)

    def forward(self, x):
        return self.d2(self.c2(F.relu(self.d1(self.c1(x.transpose(-1, -2)))))).transpose(-1, -2)


def causal_mask(n, device):
    return ~torch.tril(torch.ones(n, n, dtype=torch.bool, device=device))


# -- Model 1: SASRec Baseline (APE) --

class SASRecBaseline(nn.Module):
    def __init__(self, item_num, maxlen=50, d=64, blocks=2, heads=1, dr=0.5, **kw):
        super().__init__()
        self.d = d
        self.item_emb = nn.Embedding(item_num + 1, d, padding_idx=0)
        self.pos_emb = nn.Embedding(maxlen, d)
        self.emb_drop = nn.Dropout(dr)
        self.attn_ln = nn.ModuleList([nn.LayerNorm(d, eps=1e-8) for _ in range(blocks)])
        self.attn = nn.ModuleList([nn.MultiheadAttention(d, heads, dropout=dr, batch_first=True) for _ in range(blocks)])
        self.ffn_ln = nn.ModuleList([nn.LayerNorm(d, eps=1e-8) for _ in range(blocks)])
        self.ffn = nn.ModuleList([PointWiseFeedForward(d, dr) for _ in range(blocks)])
        self.last_ln = nn.LayerNorm(d, eps=1e-8)

    def encode(self, seqs, time_seqs=None, uf=None):
        x = self.item_emb(seqs) * math.sqrt(self.d)
        x = x + self.pos_emb(torch.arange(seqs.shape[1], device=seqs.device)).unsqueeze(0)
        x = self.emb_drop(x)
        mask = causal_mask(seqs.shape[1], seqs.device)
        for i in range(len(self.attn)):
            q = self.attn_ln[i](x)
            o, _ = self.attn[i](q, q, q, attn_mask=mask, need_weights=False)
            x = x + o
            x = x + self.ffn[i](self.ffn_ln[i](x))
        return self.last_ln(x)

    def forward(self, log, tm, pos, neg, uf=None):
        h = self.encode(log, tm, uf)
        return (h * self.item_emb(pos)).sum(-1), (h * self.item_emb(neg)).sum(-1)

    def predict(self, log, tm, items=None, uf=None):
        h = self.encode(log, tm, uf)[:, -1, :]
        if items is None:
            return torch.matmul(h, self.item_emb.weight.T)
        return self.item_emb(items).matmul(h.unsqueeze(-1)).squeeze(-1)


# -- Model 2: SASRec + RoTE --

class RoTEAttention(nn.Module):
    def __init__(self, d, heads, dr, yb, mb, db, yw, mw, dw):
        super().__init__()
        self.d, self.h, self.hd = d, heads, d // heads
        self.qkv = nn.ModuleList([nn.Linear(d, d) for _ in range(3)])
        self.out = nn.Linear(d, d)
        self.rope = MultiLevelRoTE(self.hd, yb, mb, db, yw, mw, dw)
        self.drop = nn.Dropout(dr)

    def forward(self, x, ymd, mask):
        B, S, _ = x.shape
        q, k, v = [p(x).reshape(B, S, self.h, self.hd).permute(0, 2, 1, 3) for p in self.qkv]
        q, k = self.rope(q, k, ymd, ymd)
        sc = q @ k.transpose(-2, -1) / math.sqrt(self.hd)
        if mask is not None:
            sc = sc.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        return self.out((self.drop(F.softmax(sc, -1)) @ v).permute(0, 2, 1, 3).reshape(B, S, self.d))


class SASRecRoTE(nn.Module):
    def __init__(self, item_num, maxlen=50, d=64, blocks=2, heads=1, dr=0.5,
                 yb=1e6, mb=1e4, db=100., yw=1.5, mw=1.0, dw=0.5, **kw):
        super().__init__()
        self.d = d
        self.item_emb = nn.Embedding(item_num + 1, d, padding_idx=0)
        self.emb_drop = nn.Dropout(dr)
        self.attn_ln = nn.ModuleList([nn.LayerNorm(d, eps=1e-8) for _ in range(blocks)])
        self.attn = nn.ModuleList([RoTEAttention(d, heads, dr, yb, mb, db, yw, mw, dw) for _ in range(blocks)])
        self.ffn_ln = nn.ModuleList([nn.LayerNorm(d, eps=1e-8) for _ in range(blocks)])
        self.ffn = nn.ModuleList([PointWiseFeedForward(d, dr) for _ in range(blocks)])
        self.last_ln = nn.LayerNorm(d, eps=1e-8)

    def encode(self, seqs, time_seqs, uf=None):
        x = self.item_emb(seqs) * math.sqrt(self.d)
        x = self.emb_drop(x)
        ymd = decompose_unix_timestamp(time_seqs.float())
        mask = causal_mask(seqs.shape[1], seqs.device)
        for i in range(len(self.attn)):
            q = self.attn_ln[i](x)
            x = x + self.attn[i](q, ymd, mask)
            x = x + self.ffn[i](self.ffn_ln[i](x))
        return self.last_ln(x)

    def forward(self, log, tm, pos, neg, uf=None):
        h = self.encode(log, tm, uf)
        return (h * self.item_emb(pos)).sum(-1), (h * self.item_emb(neg)).sum(-1)

    def predict(self, log, tm, items=None, uf=None):
        h = self.encode(log, tm)[:, -1, :]
        if items is None:
            return torch.matmul(h, self.item_emb.weight.T)
        return self.item_emb(items).matmul(h.unsqueeze(-1)).squeeze(-1)


# -- Model 3: SASRec + ChronoRoPE --
# KEY DESIGN: ChronoRoPE = RoTE + per-level alpha + content-aware phase shifts + temperature correction

class MultiLevelChronoRoPE(nn.Module):
    """Per-level alpha + content-aware phase shifts.
    Combines angles FIRST, then rotates ONCE to preserve norm.
    Phase shift is added ONCE to the combined angle."""
    def __init__(self, dim, yb=1e6, mb=1e4, db=100., yw=1.5, mw=1.0, dw=0.5):
        super().__init__()
        self.yb, self.mb, self.db = yb, mb, db
        self.yw, self.mw, self.dw = yw, mw, dw
        self.register_buffer("idx", torch.arange(0, dim, 2).float() / dim)

    def forward(self, q, k, ymd_q, ymd_k, alpha, phase=None):
        # alpha: (B, 3) -> per-level scaling factors
        alpha_y = alpha[:, 0].unsqueeze(-1)  # (B, 1)
        alpha_m = alpha[:, 1].unsqueeze(-1)
        alpha_d = alpha[:, 2].unsqueeze(-1)

        # Compute per-level inverse frequencies with NTK scaling
        inv_y = 1. / ((self.yb * alpha_y) ** self.idx)
        inv_m = 1. / ((self.mb * alpha_m) ** self.idx)
        inv_d = 1. / ((self.db * alpha_d) ** self.idx)

        def _combined_angles(ymd):
            """Combine angles: theta_total = w_y*theta_y + w_m*theta_m + w_d*theta_d + phase"""
            y, m, d = ymd
            ty = torch.einsum("bs,bd->bsd", y, inv_y)
            tm = torch.einsum("bs,bd->bsd", m, inv_m)
            td = torch.einsum("bs,bd->bsd", d, inv_d)
            # Combine angles BEFORE cos/sin (preserves rotation norm)
            combined = self.yw * ty + self.mw * tm + self.dw * td
            # Add phase shift ONCE to the combined angle
            if phase is not None:
                combined = combined + phase
            return combined.cos().unsqueeze(1), combined.sin().unsqueeze(1)

        qc, qs = _combined_angles(ymd_q)
        if ymd_q is ymd_k:
            kc, ks = qc, qs
        else:
            kc, ks = _combined_angles(ymd_k)
        return _apply_rotary(q, qc, qs), _apply_rotary(k, kc, ks)

class ChronoAttention(nn.Module):
    """Attention with per-level alpha, CARoPE phase shifts, and YaRN temperature."""
    def __init__(self, d, heads, dr, yb, mb, db, yw, mw, dw):
        super().__init__()
        self.d, self.h, self.hd = d, heads, d // heads
        self.qkv = nn.ModuleList([nn.Linear(d, d) for _ in range(3)])
        self.out = nn.Linear(d, d)
        self.rope = MultiLevelChronoRoPE(self.hd, yb, mb, db, yw, mw, dw)
        self.drop = nn.Dropout(dr)
        # CARoPE: content-aware phase shift (init near zero -> starts as standard RoTE)
        self.phase_proj = nn.Linear(d, self.hd // 2, bias=False)
        nn.init.zeros_(self.phase_proj.weight)

    def forward(self, x, ymd, alpha, mask):
        B, S, _ = x.shape
        q, k, v = [p(x).reshape(B, S, self.h, self.hd).permute(0, 2, 1, 3) for p in self.qkv]
        # Content-aware phase offset from item representations
        phase = self.phase_proj(x)  # (B, S, hd//2)
        q, k = self.rope(q, k, ymd, ymd, alpha, phase)
        # YaRN-style temperature: compensate variance shift from base scaling
        temp = alpha.mean(dim=-1).view(B, 1, 1, 1).sqrt()
        sc = q @ k.transpose(-2, -1) / (math.sqrt(self.hd) * temp)
        if mask is not None:
            sc = sc.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        return self.out((self.drop(F.softmax(sc, -1)) @ v).permute(0, 2, 1, 3).reshape(B, S, self.d))

class SASRecChronoRoPE(nn.Module):
    def __init__(self, item_num, maxlen=50, d=64, blocks=2, heads=1, dr=0.5,
                 meta_input_dim=9, meta_hidden_dims=None, meta_eps=0.1, meta_init_alpha=1.0,
                 diversity_lambda=0.01,
                 yb=1e6, mb=1e4, db=100., yw=1.5, mw=1.0, dw=0.5, **kw):
        super().__init__()
        self.d = d
        self.diversity_lambda = diversity_lambda
        self.item_emb = nn.Embedding(item_num + 1, d, padding_idx=0)
        self.emb_drop = nn.Dropout(dr)
        self.meta_mlp = MetaMLP(meta_input_dim, meta_hidden_dims, meta_eps, meta_init_alpha)
        self.attn_ln = nn.ModuleList([nn.LayerNorm(d, eps=1e-8) for _ in range(blocks)])
        self.attn = nn.ModuleList([ChronoAttention(d, heads, dr, yb, mb, db, yw, mw, dw) for _ in range(blocks)])
        self.ffn_ln = nn.ModuleList([nn.LayerNorm(d, eps=1e-8) for _ in range(blocks)])
        self.ffn = nn.ModuleList([PointWiseFeedForward(d, dr) for _ in range(blocks)])
        self.last_ln = nn.LayerNorm(d, eps=1e-8)
        self._last_alpha = None

    @property
    def last_alpha_u(self):
        return self._last_alpha

    def encode(self, seqs, time_seqs, uf):
        x = self.item_emb(seqs) * math.sqrt(self.d)
        x = self.emb_drop(x)
        alpha = self.meta_mlp(uf)
        self._last_alpha = alpha.detach()
        mask = causal_mask(seqs.shape[1], seqs.device)
        
        # Times are already globally relative per-user from the dataset
        ymd = decompose_unix_timestamp(time_seqs.float())
        for i in range(len(self.attn)):
            q = self.attn_ln[i](x)
            x = x + self.attn[i](q, ymd, alpha, mask)
            x = x + self.ffn[i](self.ffn_ln[i](x))
        return self.last_ln(x)

    def forward(self, log, tm, pos, neg, uf):
        h = self.encode(log, tm, uf)
        pl = (h * self.item_emb(pos)).sum(-1)
        nl = (h * self.item_emb(neg)).sum(-1)
        return pl, nl, MetaMLP.diversity_loss(self._last_alpha)

    def predict(self, log, tm, items=None, uf=None):
        h = self.encode(log, tm, uf)[:, -1, :]
        if items is None:
            return torch.matmul(h, self.item_emb.weight.T)
        return self.item_emb(items).matmul(h.unsqueeze(-1)).squeeze(-1)

## SECTION 5: EVALUATION

In [ ]:
def evaluate(model, sampler, mode="test", topk=[5, 10], full_eval=True):
    model.eval()
    all_ranks = []
    all_densities = []
    with torch.no_grad():
        for uids, log, tm, uf, batch_targets in sampler.eval_batches(mode, full_eval=full_eval):
            log, tm, uf = log.to(DEVICE), tm.to(DEVICE), uf.to(DEVICE)
            if full_eval:
                scores = model.predict(log, tm, items=None, uf=uf)  # (B, num_items + 1)
                for i, (target, hist_items) in enumerate(batch_targets):
                    scores[i, 0] = float("-inf")
                    mask_items = list(hist_items - {target})
                    if mask_items:
                        scores[i, mask_items] = float("-inf")
                    target_score = scores[i, target].item()
                    rank = (scores[i, 1:] > target_score).sum().item() + 1
                    all_ranks.append(rank)
                    all_densities.append(uf[i, 0].item())
            else:
                cands = batch_targets.to(DEVICE)
                scores = model.predict(log, tm, cands, uf)
                _, idx = scores.sort(dim=-1, descending=True)
                for i in range(scores.shape[0]):
                    all_ranks.append((idx[i] == 0).nonzero(as_tuple=True)[0].item() + 1)
                    all_densities.append(uf[i, 0].item())
                
    sparse_ranks, normal_ranks, dense_ranks = [], [], []
    for rank, density in zip(all_ranks, all_densities):
        if density < -0.5: sparse_ranks.append(rank)
        elif density > 0.5: dense_ranks.append(rank)
        else: normal_ranks.append(rank)
        
    ranks_dict = {
        "Overall": np.array(all_ranks),
        "Sparse": np.array(sparse_ranks) if sparse_ranks else np.array([]),
        "Normal": np.array(normal_ranks) if normal_ranks else np.array([]),
        "Dense": np.array(dense_ranks) if dense_ranks else np.array([]),
    }

    metrics = {}
    for group, ranks in ranks_dict.items():
        if len(ranks) == 0: continue
        prefix = f"{group}_" if group != "Overall" else ""
        for k in topk:
            metrics[f"{prefix}R@{k}"] = float(np.mean(ranks <= k))
            metrics[f"{prefix}N@{k}"] = float(np.mean(np.where(ranks <= k, 1.0 / np.log2(ranks + 1), 0.0)))
    return metrics

## SECTION 6: TRAINING LOOP

In [ ]:
def train_model(model_name, model, sampler, config):
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"  Device: {DEVICE}")
    print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")
    if hasattr(model, 'meta_mlp'):
        mp = sum(p.numel() for p in model.meta_mlp.parameters())
        print(f"  Meta-MLP params: {mp:,}")
    print(f"{'='*60}\n")

    model = model.to(DEVICE)
    
    # Match official RoTE parameter initialization
    # BUG-2/4 FIX: Skip meta_mlp and phase_proj to preserve careful initialization
    for name, param in model.named_parameters():
        if 'meta_mlp' in name or 'phase_proj' in name:
            continue  # preserve MetaMLP init (~1.0 alpha) and phase_proj zero init
        try:
            torch.nn.init.xavier_normal_(param.data)
        except Exception:
            pass
    if hasattr(model, "item_emb"):
        model.item_emb.weight.data[0, :] = 0

    # Same optimizer for all models
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], betas=(0.9, 0.98))

    best_ndcg, best_epoch = 0.0, 0
    best_state = None
    history = []

    for epoch in range(1, config["num_epochs"] + 1):
        model.train()
        total_loss, n_batch = 0.0, 0
        alphas = []
        t0 = time.time()

        for batch in sampler.train_epoch():
            log, tm, pos, neg, uf = [b.to(DEVICE) for b in batch]

            if model_name == "ChronoRoPE":
                pl, nl, div = model(log, tm, pos, neg, uf)
            else:
                pl, nl = model(log, tm, pos, neg, uf)
                div = torch.tensor(0.0, device=DEVICE)

            pos_labels = torch.ones(pl.shape, device=DEVICE)
            neg_labels = torch.zeros(nl.shape, device=DEVICE)
            indices = torch.where(pos != 0)

            bce_fn = torch.nn.BCEWithLogitsLoss()
            bce_loss = bce_fn(pl[indices], pos_labels[indices]) + bce_fn(nl[indices], neg_labels[indices])
            loss = bce_loss + config["diversity_lambda"] * div

            optimizer.zero_grad()
            loss.backward()

            if config["grad_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            total_loss += loss.item()
            n_batch += 1
            if model_name == "ChronoRoPE" and model.last_alpha_u is not None:
                alphas.append(model.last_alpha_u.cpu().numpy())

        dt = time.time() - t0
        avg = total_loss / max(n_batch, 1)
        msg = f"  Epoch {epoch:3d}/{config['num_epochs']} ({dt:.1f}s) | Loss: {avg:.4f}"

        if model_name == "ChronoRoPE" and alphas:
            a = np.concatenate(alphas, axis=0)
            msg += f" | a_y={a[:,0].mean():.2f} a_m={a[:,1].mean():.2f} a_d={a[:,2].mean():.2f}"

        if epoch % config["eval_interval"] == 0 or epoch == config["num_epochs"]:
            val = evaluate(model, sampler, "val", config["topk"], full_eval=config.get("full_eval", True))
            # evaluate() returns keys like 'N@10', 'R@5' — no 'val_' prefix
            ndcg_key = f"N@{config['topk'][-1]}"
            ndcg = val.get(ndcg_key, 0.0)
            msg += f" | Val N@{config['topk'][-1]}: {ndcg:.4f}"

            # Store in history WITH 'val_' prefix
            record = {"epoch": epoch, "loss": avg, **{f"val_{k}": v for k, v in val.items()}}
            if model_name == "ChronoRoPE" and alphas:
                a = np.concatenate(alphas, axis=0)
                record["alpha_y_mean"] = float(a[:, 0].mean())
                record["alpha_m_mean"] = float(a[:, 1].mean())
                record["alpha_d_mean"] = float(a[:, 2].mean())
            history.append(record)

            if ndcg > best_ndcg:
                best_ndcg = ndcg
                best_epoch = epoch
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                msg += " *"

        print(msg)

    # Final test with best model
    if best_state is not None:
        print(f"\n  Loading best model (epoch {best_epoch}) ...")
        model.load_state_dict(best_state)
    else:
        print(f"\n  No improvement found, using last epoch model.")

    test = evaluate(model, sampler, "test", config["topk"], full_eval=config.get("full_eval", True))
    print(f"\n  {'─'*40}")
    print(f"  TEST RESULTS for {model_name}:")
    for k, v in test.items():
        print(f"    {k}: {v:.4f}")
    print(f"  {'─'*40}\n")

    return {"model_name": model_name, "test": test, "best_epoch": best_epoch, "history": history}

## SECTION 7: MAIN — RUN ALL EXPERIMENTS

In [ ]:
def main():
    set_seed(CONFIG["seed"])
    print("=" * 60)
    print("  Personalized ChronoRoPE Experiment")
    print("=" * 60)

    # Download & load data
    print("\nLoading dataset ...")
    data_path = download_amazon_dataset("Toys_and_Games")
    dataset = load_data(data_path)
    print(f"  Users: {dataset.num_users:,} | Items: {dataset.num_items:,}")
    print(f"  Avg seq len: {np.mean([len(s) for s in dataset.user_seqs.values()]):.1f}")

    sampler = TrainSampler(dataset, CONFIG["maxlen"], CONFIG["batch_size"], CONFIG["num_neg_eval"])

    # Build all 3 models
    common = dict(item_num=dataset.num_items, maxlen=CONFIG["maxlen"],
                  d=CONFIG["hidden_units"], blocks=CONFIG["num_blocks"],
                  heads=CONFIG["num_heads"], dr=CONFIG["dropout_rate"])

    models = {
        "Baseline": SASRecBaseline(**common),
        "RoTE": SASRecRoTE(**common,
            yb=CONFIG["year_base"], mb=CONFIG["month_base"], db=CONFIG["day_base"],
            yw=CONFIG["year_weight"], mw=CONFIG["month_weight"], dw=CONFIG["day_weight"]),
        "ChronoRoPE": SASRecChronoRoPE(**common,
            meta_input_dim=CONFIG["meta_input_dim"], meta_hidden_dims=CONFIG["meta_hidden_dims"],
            meta_eps=CONFIG["meta_eps"], meta_init_alpha=CONFIG["meta_init_alpha"],
            diversity_lambda=CONFIG["diversity_lambda"],
            yb=CONFIG["year_base"], mb=CONFIG["month_base"], db=CONFIG["day_base"],
            yw=CONFIG["year_weight"], mw=CONFIG["month_weight"], dw=CONFIG["day_weight"]),
    }

    # Train all
    all_results = {}
    for name, model in models.items():
        set_seed(CONFIG["seed"])  # Reset seed for fair comparison
        result = train_model(name, model, sampler, CONFIG)
        all_results[name] = result

    # -- Summary Table --
    print("\n" + "=" * 60)
    print("  FINAL COMPARISON")
    print("=" * 60)
    r_key = "R@5" if "R@5" in list(all_results.values())[0]["test"] else "HR@5"
    n_key = "N@5" if "N@5" in list(all_results.values())[0]["test"] else "NDCG@5"
    r10_key = "R@10" if "R@10" in list(all_results.values())[0]["test"] else "HR@10"
    n10_key = "N@10" if "N@10" in list(all_results.values())[0]["test"] else "NDCG@10"
    
    print(f"\n  {'Model':<15} | {r_key:>7} | {r10_key:>7} | {n_key:>7} | {n10_key:>7}")
    print(f"  {'-'*15}-+-{'-'*7}-+-{'-'*7}-+-{'-'*7}-+-{'-'*7}")
    for name, r in all_results.items():
        t = r["test"]
        marker = " *" if name == "ChronoRoPE" else ""
        print(f"  {name:<15} | {t[r_key]:>7.4f} | {t[r10_key]:>7.4f} | {t[n_key]:>7.4f} | {t[n10_key]:>7.4f}{marker}")
    print()

    # -- Visualization --
    try:
        import matplotlib.pyplot as plt

        # Plot training curves
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for name, r in all_results.items():
            epochs = [h["epoch"] for h in r["history"]]
            losses = [h["loss"] for h in r["history"]]
            # History stores keys with 'val_' prefix (e.g. 'val_N@10')
            ndcg_key = f"val_N@{CONFIG['topk'][-1]}"
            ndcgs = [h.get(ndcg_key, 0.0) for h in r["history"]]
            ls = "--" if name == "Baseline" else ("-." if name == "RoTE" else "-")
            axes[0].plot(epochs, losses, label=name, linestyle=ls)
            axes[1].plot(epochs, ndcgs, label=name, linestyle=ls)

        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Training Loss")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel(f"NDCG@{CONFIG['topk'][-1]}"); axes[1].set_title("Validation NDCG")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig("training_curves.png", dpi=150)
        plt.show()
        print("  Training curves saved to training_curves.png")

        # alpha_u scatter plot -- per-level
        if "ChronoRoPE" in all_results:
            chrono_model = models["ChronoRoPE"].to(DEVICE)
            chrono_model.eval()
            densities, ay_out, am_out, ad_out = [], [], [], []
            with torch.no_grad():
                for uid in sampler.users:
                    uf = torch.tensor(dataset.user_features[uid], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                    a = chrono_model.meta_mlp(uf).squeeze(0).cpu().numpy()  # (3,)
                    densities.append(dataset.user_features[uid][0])
                    ay_out.append(a[0]); am_out.append(a[1]); ad_out.append(a[2])

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            for ax, vals, name in zip(axes, [ay_out, am_out, ad_out], ["alpha_Year", "alpha_Month", "alpha_Day"]):
                ax.scatter(densities, vals, alpha=0.3, s=10, c=densities, cmap="viridis")
                ax.set_xlabel("User Temporal Density")
                ax.set_ylabel(f"Learned {name}")
                ax.set_title(f"{name} vs Density")
                ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig("alpha_scatter.png", dpi=150)
            plt.show()
            print("  Per-level alpha scatter plots saved to alpha_scatter.png")

    except ImportError:
        print("  (matplotlib not available, skipping plots)")

    print("\nExperiment complete!")
    return all_results


if __name__ == "__main__":
    results = main()